In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Integer Square Root

In this notebook we work with very big numbers.  Therefore, we use the module `Z` of the library Zarith, which implements integers of arbitrary size.  Their type is `Z.t`.

In [ ]:
#require "zarith.top";;

The function `isqrt` takes one argument:
- `a` is a natural number that is less than $2^{128\,000}$.

It returns the largest natural number $r$ such that
$r^2 \leq a$, i.e. we have
$$ \texttt{isqrt}(a) := \max\bigl(\{ r \in \mathbb{N} \mid r^2 \leq a \}\bigr). $$
If `a` is not less than $2^{128\,000}$, the exception `Invalid_argument` is raised.

**Implementation:** Since $a < 2^{128\,000}$, we know that `isqrt a` is less than $2^{64\,000}$.
The implementation computes the result bit by bit starting from the bit with the highest value.  The reference `p` runs through the powers $2^{64\,000}, 2^{63\,999}, \cdots, 2^1, 2^0$.  For every `p` we check whether we can add `p` to `r` without the square of `r` exceeding `a`.  If this is the case, the bit `p` is set in `r`.

Inside the *local open* `Z.(...)`, the arithmetic operators `+`, `/`, and `**` refer to the integers of type `Z.t`, and `~$` converts an `int` into a `Z.t`.  The comparison operators are not redefined there.  Therefore, we compare numbers of type `Z.t` with the functions `Z.geq` ($\geq$) and `Z.gt` ($>$).  `Z.zero` is the number $0$.

In [ ]:
let isqrt (a : Z.t) : Z.t =
  if Z.geq a Z.(~$2 ** 128_000) then
    invalid_arg "isqrt: the argument must be less than 2^128000";
  let r = ref Z.zero in
  let p = ref Z.(~$2 ** 64_000) in
  while Z.gt !p Z.zero do
    if Z.geq a Z.((!r + !p) ** 2) then
      r := Z.(!r + !p);
    p := Z.(!p / ~$2)
  done;
  !r

We have
$$\sqrt{2 \cdot 10^{2 \cdot k}} = \sqrt{2} \cdot 10^k.$$
Hence the digits of $\texttt{isqrt}(2 \cdot 10^{2 \cdot k})$ are the first $k+1$ digits of $\sqrt{2}$.  We compute $10\,000$ decimal places of $\sqrt{2}$ and insert a decimal point after the first digit.  The exponent `e` is computed outside of `Z.(...)`, since inside, the operator `*` multiplies numbers of type `Z.t`, not of type `int`.  The variables `e`, `r`, and `s` are defined locally with `let ... in`.  Otherwise, the toplevel would display the value of `r`, which has more than $10\,000$ digits.  The function `Z.to_string` converts a number into the string of its decimal digits and `String.sub s i l` returns the substring of `s` that starts at index `i` and has the length `l`.

The function `time`, which is defined in the file `setup.ml`, takes two arguments:
- `label` is a string,
- `f` is a function whose only argument is `()`.

It computes `f ()`, prints `label` together with the time this computation took, and returns the result of `f ()`.  The computation to be measured is wrapped into an anonymous function `fun () -> ...`, so that it is only executed when `time` calls `f`.

In [ ]:
let num_digits = 10_000;;
let e = 2 * num_digits in
let r = time (Printf.sprintf "Computation of %d digits of √2" num_digits)
             (fun () -> isqrt Z.(~$2 * ~$10 ** e)) in
let s = Z.to_string r in
print_endline (String.sub s 0 1 ^ "." ^ String.sub s 1 (String.length s - 1))

We can check this at [CATONMAT](https://catonmat.net/tools/generate-sqrt2-digits).

When we compare this with the floating point implementation of the function `sqrt`, we see that only the first 15 decimal places of the floating point result are correct: The floating point number closest to $\sqrt{2}$ is a bit bigger than $\sqrt{2}$.

In [ ]:
sqrt 2.0